# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [5]:
%%writefile ../schema/student.thrift

struct Student {
    1: required string name,
    2: required i32 age,
    3: required list<string> courses
}

service School {
    Student enrollCourse(1: required Student student, 2: required string course)
}

Overwriting ../schema/student.thrift


**Thrift server (student_server.py)**

In [6]:
%%writefile ../student_thrift_server.py
import thriftpy2
student_thrift = thriftpy2.load("./schema/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def enrollCourse(self, student, course):
        student.courses.append(course)
        return student

server = make_server(student_thrift.School, School(), client_timeout=None)
print("Server started...")
print("Press Ctrl+C to stop the server.")
server.serve()

Overwriting ../student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [7]:
import thriftpy2
student_thrift = thriftpy2.load("../schema/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_client

# Connect to the server
school = make_client(student_thrift.School, timeout=None)

# Create a student
leo = student_thrift.Student(
    name="Leo", age=15, courses=["Math", "Chemistry"]
)
print(f"Initial courses: {leo.courses}")

# Enroll Leo in a new course
leo = school.enrollCourse(leo, "Computer Science")
print(f"Updated courses: {leo.courses}")


Initial courses: ['Math', 'Chemistry']
Updated courses: ['Math', 'Chemistry', 'Computer Science']


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [17]:
%%writefile book.proto
syntax = "proto3";

message Book {
  string title = 1;
  string author = 2;
  int32 page_count = 3;
}

service Library {
  rpc checkoutBook(Book) returns (Book) {}
}

Overwriting book.proto


Run

**Protobuf server (book_server.py)**

In [18]:
%%writefile book_server.py
from concurrent import futures
import grpc
import book_pb2_grpc

class Library(book_pb2_grpc.LibraryServicer):
    def checkoutBook(self, request, context):
        return request

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(Library(), server)
server.add_insecure_port('[::]:50051')
print("Server started...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()

Overwriting book_server.py


Run

**Protobuf client (book_client.py)**

In [19]:
import grpc
import book_pb2
import book_pb2_grpc

with grpc.insecure_channel('localhost:50051') as channel:
    book = book_pb2.Book(
        title="I May Be Wrong",
        author="Björn Natthiko Lindeblad",
        page_count=257
    )
    print(f"Checking out: '{book.title}' by {book.author} ({book.page_count} pages)")

    stub = book_pb2_grpc.LibraryStub(channel)
    checked_out_book = stub.checkoutBook(book)
    print(f"Checked out: '{checked_out_book.title}' by {checked_out_book.author} ({checked_out_book.page_count} pages)")
    

Checking out: 'I May Be Wrong' by Björn Natthiko Lindeblad (257 pages)
Checked out: 'I May Be Wrong' by Björn Natthiko Lindeblad (257 pages)
